# Building a Quantitative Trading Strategy
## Part 3: Live Trading Implementation

Este notebook demuestra cómo implementar un sistema de trading en tiempo real.
We'll build streaming data structures, define trading APIs, and execute
trades on a simulated exchange.

**Repository:** https://github.com/memlabs-research/build-a-quant-trading-strategy

## Trading System Overview

Nuestro pipeline completo:

```
y_hat = model(x)          # Paso 1: Generar predicciones (Part 1)
orders = strategy(y_hat)  # Step 2: Convert to orders (Part 2)
execute(orders)           # Paso 3: Ejecutar operaciones (Esta Parte)
```

**Objetivos de la Parte 3:**
1. Construir estructuras de datos en streaming para procesamiento en tiempo real
2. Design a scalable trading system architecture
3. Execute trades on a simulated exchange

---
## Resumen: Nuestro Modelo Entrenado

De la Parte 1, tenemos un modelo lineal AR(3) que predice log returns con 12 horas de anticipación.

In [1]:

import torch

# Importar de nuestra librería quant research
from src.quant_research import (
    LinearModel,
    print_model_params,
)

# Cargar el modelo entrenado
model = LinearModel(3)
model.load_state_dict(torch.load('data/models/model_weights.pth', weights_only=True))
model.eval()

LinearModel(
  (linear): Linear(in_features=3, out_features=1, bias=True)
)

### Model Summary

El modelo usa 3 log returns con lag para predecir el siguiente retorno de 12 horas.


In [2]:

print_model_params(model)

# **Resumen de la Parte 2:**
# - ~14% de retorno sin optimización
# - With compounding + leverage → >40% return

linear.weight:
[[-0.09520352 -0.03124688 -0.0465919 ]]
linear.bias:
[0.00107722]


---
## 1. Streaming Data Structures

### 1.1 La Interfaz Tick

El bloque fundamental de nuestro sistema de streaming es la interfaz `Tick`.
It represents a single update in a data stream.

```
on_tick(value) → result
```

Este patrón nos permite componer operaciones de streaming complejas a partir de primitivas simples.

In [3]:

from abc import ABC, abstractmethod
from typing import Generic, TypeVar, Optional

T = TypeVar('T')  # Input type
R = TypeVar('R')  # Output type


class Tick(ABC, Generic[T, R]):
    """
    Abstract base class for streaming data processors.

    A Tick receives an input value and optionally produces an output.
    This enables composable, modular stream processing.
    """

    @abstractmethod
    def on_tick(self, val: T) -> R:
        """Process a new value and optionally return a result."""
        pass

---
## 2. Sliding Window Implementations

Las ventanas deslizantes son esenciales para análisis de series temporales. Mantienen
un buffer de tamaño fijo con los valores más recientes.

### 2.1 Deque-Based Window

Usa `collections.deque` de Python para operaciones eficientes O(1) de append/pop.

In [4]:

from collections import deque
from typing import Deque
import numpy as np


class DequeWindow(Tick[T, Optional[T]], Generic[T]):
    """
    A fixed-size sliding window using collections.deque.

    Properties:
    - O(1) append and pop operations
    - Automatic size management
    - Memory efficient

    Example:
        >>> window = DequeWindow(3)
        >>> window.on_tick(1)  # None (not full yet)
        >>> window.on_tick(2)  # None
        >>> window.on_tick(3)  # None (just became full)
        >>> window.on_tick(4)  # Returns 1 (dropped oldest)
    """

    def __init__(self, n: int):
        self._data: Deque[T] = deque(maxlen=n)

    def on_tick(self, val: T) -> Optional[T]:
        """Append value and return dropped value (if window was full)."""
        dropped = None
        if self.is_full():
            dropped = self._data[0]
        self._data.append(val)
        return dropped

    def is_full(self) -> bool:
        """Check if window has reached capacity."""
        return self._data.maxlen == len(self._data)

    def append_left(self, val: T) -> Optional[T]:
        """Append to the left (front) of the window."""
        dropped = None
        if self.is_full():
            dropped = self._data[-1]
        self._data.appendleft(val)
        return dropped

    def to_numpy(self) -> np.ndarray:
        """Convert window contents to numpy array."""
        return np.array(self._data)

    def __repr__(self) -> str:
        return f"DequeWindow(capacity={self._data.maxlen}, values={list(self._data)})"

### 2.2 DequeWindow Examples

In [5]:

# Create a window of size 3
w = DequeWindow(3)
print(f"Empty window: {w}")

# Llenar la ventana
w.on_tick(1)
w.on_tick(2)
w.on_tick(3)
print(f"Full window: {w}")

# Add one more (drops oldest)
dropped = w.on_tick(4)
print(f"After adding 4, dropped: {dropped}")
print(f"Current window: {w}")

# La ventana maneja eficientemente millones de actualizaciones
w = DequeWindow(3)
for i in range(1_000_000):
    w.on_tick(i)
print(f"After 1M updates: {w}")

Empty window: DequeWindow(capacity=3, values=[])
Full window: DequeWindow(capacity=3, values=[1, 2, 3])
After adding 4, dropped: 1
Current window: DequeWindow(capacity=3, values=[2, 3, 4])
After 1M updates: DequeWindow(capacity=3, values=[999997, 999998, 999999])


### 2.3 Ventana Basada en NumPy

Para operaciones numéricas, una implementación basada en NumPy puede ser más rápida
due to contiguous memory and vectorization.

In [6]:


class NumpyWindow(Tick[T, Optional[T]]):
    """
    A fixed-size sliding window using NumPy arrays.

    Properties:
    - Contiguous memory layout
    - Better for numerical operations
    - Slightly slower append due to shifting

    Use this when you need to perform vectorized operations on the window.
    """

    def __init__(self, n: int, dtype=np.float64):
        if n <= 0:
            raise ValueError("Capacity must be positive.")
        self._capacity = n
        self._data = np.zeros(n, dtype=dtype)
        self._size = 0

    def on_tick(self, val: float) -> Optional[float]:
        """Append value and return dropped value (if window was full)."""
        dropped = None

        if self._size < self._capacity:
            self._data[self._size] = val
            self._size += 1
        else:
            dropped = self._data[0]
            # Shift left in-place
            for i in range(1, self._capacity):
                self._data[i - 1] = self._data[i]
            self._data[-1] = val

        return dropped

    def __getitem__(self, idx: int) -> float:
        """Index access (0 = oldest)."""
        if not 0 <= idx < self._size:
            raise IndexError("Index out of range.")
        return self._data[idx]

    def __len__(self) -> int:
        return self._size

    def capacity(self) -> int:
        return self._capacity

    def is_full(self) -> bool:
        return self._size == self._capacity

    def values(self) -> np.ndarray:
        """Return valid portion of the array."""
        return self._data[:self._size]

    def __repr__(self) -> str:
        vals = self.values().tolist()
        return f"NumpyWindow(capacity={self._capacity}, size={self._size}, values={vals})"

### 2.4 Performance Comparison

Let's benchmark both implementations.

In [7]:


def benchmark_window(window, n):
    """Benchmark window by adding n elements."""
    for i in range(n):
        window.on_tick(i)


window_size = 10
n = 5_000_000

print(f"\nBenchmarking {n:,} updates with window size {window_size}:")

# Nota: Ejecutar en Jupyter con %%time magic para medición precisa
# NumpyWindow: ~2.5 seconds
# DequeWindow: ~0.5 segundos (más rápido para operaciones puras de append)


Benchmarking 5,000,000 updates with window size 10:


---
## 3. Streaming Value Trackers

### 3.1 Last Value Tracker

Rastreador simple que siempre retorna el valor más reciente.

In [8]:


class Last(Tick[T, T], Generic[T]):
    """
    Tracks the last value received.

    Useful for maintaining current state (e.g., last price, last signal).
    """

    def __init__(self):
        self._value: Optional[T] = None

    def on_tick(self, val: T) -> Optional[T]:
        """Update and return the new value."""
        self._value = val
        return val

    def __repr__(self) -> str:
        return f"Last(value={self._value})"


# Example
last_val = Last()
print(f"Initial: {last_val}")

for i in range(5):
    last_val.on_tick(i)
print(f"After 5 updates: {last_val}")

Initial: Last(value=None)
After 5 updates: Last(value=4)


---
## 4. Streaming Log Returns

### 4.1 Log Return Definition

Recordemos la fórmula del log return:

$$r_t = \log\left(\frac{P_t}{P_{t-1}}\right)$$

**Propiedad:** Los log returns son aditivos en el tiempo.

In [9]:

# Example calculation
ts = [100, 120, 100]
log_returns = [
    np.log(ts[1] / ts[0]),  # +18.2%
    np.log(ts[2] / ts[1]),  # -18.2%
]
print(f"Log returns: {log_returns}")
print(f"Total log return: {np.sum(log_returns):.4f} (back to start)")

Log returns: [np.float64(0.1823215567939546), np.float64(-0.1823215567939546)]
Total log return: 0.0000 (back to start)


### 4.2 Streaming Log Return Calculator

In [10]:


class LogReturn(Tick[float, Optional[float]], Generic[T]):
    """
    Streaming log return calculator.

    Maintains a window of the last 2 prices to compute log returns.

    Formula: log(P_t / P_{t-1})

    Example:
        >>> lr = LogReturn()
        >>> lr.on_tick(100.0)  # None (need 2 prices)
        >>> lr.on_tick(120.0)  # Returns 0.1823 (log(120/100))
    """

    def __init__(self):
        self._window = NumpyWindow(2)

    def on_tick(self, val: float) -> Optional[float]:
        """Add price and return log return (if we have 2 prices)."""
        self._window.on_tick(val)
        if self._window.is_full():
            return np.log(self._window[1] / self._window[0])
        else:
            return None

    def __repr__(self) -> str:
        return f"LogReturn(window={self._window})"


# Example
f = LogReturn()
print(f"Initial: {f}")

f.on_tick(100.0)
print(f"After first price: {f}")

v = f.on_tick(120.0)
print(f"Log return 100→120: {v:.4f}")

v = f.on_tick(100.0)
print(f"Log return 120→100: {v:.4f}")

Initial: LogReturn(window=NumpyWindow(capacity=2, size=0, values=[]))
After first price: LogReturn(window=NumpyWindow(capacity=2, size=1, values=[100.0]))
Log return 100→120: 0.1823
Log return 120→100: -0.1823


---
## 5. Streaming Feature Generator

### 5.1 Lagged Log Returns

Nuestro modelo necesita los últimos 3 log returns como features.
Creamos un componente de streaming que mantiene este estado.

In [11]:


class LogReturnLags(Tick[float, torch.Tensor]):
    """
    Streaming lagged log returns generator.

    Computes log returns from prices and maintains a window of the
    last N log returns as model features.

    Args:
        no_lags: Number of lag features to maintain

    Example:
        >>> lags = LogReturnLags(3)
        >>> # Feed prices until we have enough history
        >>> for price in [90, 100, 150, 110, 160]:
        ...     features = lags.on_tick(price)
        >>> features  # tensor([lag_1, lag_2, lag_3])
    """

    def __init__(self, no_lags: int):
        self._lags = DequeWindow(no_lags)
        self._log_return = LogReturn()

    def on_tick(self, val: float) -> Optional[torch.Tensor]:
        """Add price and return features tensor (when ready)."""
        log_ret = self._log_return.on_tick(val)
        if log_ret is not None:
            # Agregar a la izquierda para que lag_1 sea el más reciente
            self._lags.append_left(log_ret)
            if self._lags.is_full():
                return torch.tensor(self._lags.to_numpy(), dtype=torch.float32)
        return None

    def __repr__(self) -> str:
        return f"LogReturnLags(lags={self._lags}, log_return={self._log_return})"

### 5.2 LogReturnLags Example

In [12]:

lags = LogReturnLags(3)
print(f"Initial: {lags}")

# Feed prices one by one
prices = [90, 100, 150, 110, 160]
for price in prices:
    result = lags.on_tick(price)
    print(f"Price {price}: features = {result}")

# Verificar el cálculo
print(f"\nExpected lags:")
print(f"  lag_1: log(160/110) = {np.log(160/110):.4f}")
print(f"  lag_2: log(110/150) = {np.log(110/150):.4f}")
print(f"  lag_3: log(150/100) = {np.log(150/100):.4f}")

Initial: LogReturnLags(lags=DequeWindow(capacity=3, values=[]), log_return=LogReturn(window=NumpyWindow(capacity=2, size=0, values=[])))
Price 90: features = None
Price 100: features = None
Price 150: features = None
Price 110: features = tensor([-0.3102,  0.4055,  0.1054])
Price 160: features = tensor([ 0.3747, -0.3102,  0.4055])

Expected lags:
  lag_1: log(160/110) = 0.3747
  lag_2: log(110/150) = -0.3102
  lag_3: log(150/100) = 0.4055


### 5.3 Inferencia de Modelo en Streaming

Ahora podemos alimentar features de streaming directamente a nuestro modelo.

In [13]:

lags = LogReturnLags(3)
lags.on_tick(90)
lags.on_tick(100)
lags.on_tick(150)
lags.on_tick(110)
features = lags.on_tick(160)

# Run inference
X = features
with torch.no_grad():
    y_hat = model(X)

print(f"\nModel prediction: {y_hat.item():.6f}")
print(f"Prediction direction: {'LONG' if y_hat.item() > 0 else 'SHORT'}")


Model prediction: -0.043795
Prediction direction: SHORT


---
## 6. Trading System Architecture

### 6.1 Usando Decimal para Dinero

**Importante:** ¡Nunca uses floats para dinero debido a problemas de precisión!

In [14]:

val = 0.1
total = 0.0
for i in range(10):
    total += val
print(f"Float precision error: 10 × 0.1 = {total}")

# Use Decimal instead
from decimal import Decimal

dp = Decimal('0.2')
val = Decimal(0.1).quantize(dp)
total = Decimal(0.0).quantize(dp)
for i in range(10):
    total += val
print(f"Decimal precision: 10 × 0.1 = {total}")

Float precision error: 10 × 0.1 = 0.9999999999999999
Decimal precision: 10 × 0.1 = 1.0


### 6.2 Order Data Class

In [15]:


from dataclasses import dataclass


@dataclass(frozen=True)
class Order:
    """
    Immutable order representation.

    Attributes:
        sym: Trading symbol (e.g., 'BTCUSDT')
        signed_qty: Positive = buy, Negative = sell
    """
    sym: str
    signed_qty: Decimal

    def __str__(self) -> str:
        sign = "LONG" if self.signed_qty > 0 else "SHORT"
        return f"Order({sign} {abs(self.signed_qty)} {self.sym})"

### 6.3 Trade Data Class

In [16]:


def decimal_sign(d: Decimal) -> int:
    """Return the sign of a Decimal value."""
    return 1 if d > Decimal(0) else -1


def is_long(x: Decimal) -> bool:
    """Check if quantity represents a long position."""
    return decimal_sign(x) > 0


@dataclass(frozen=True)
class Trade:
    """
    Immutable trade execution record.

    Attributes:
        sym: Trading symbol
        signed_qty: Executed quantity (positive = buy)
        price: Execution price
        pnl: Realized P&L from closing a position
    """
    sym: str
    signed_qty: Decimal
    price: Decimal
    pnl: Decimal

    def __str__(self) -> str:
        sign = "LONG" if is_long(self.signed_qty) else "SHORT"
        return f"Trade({sign} {abs(self.signed_qty)} {self.sym} @ {self.price}, PnL: {self.pnl})"

    def is_long(self) -> bool:
        return is_long(self.signed_qty)

### 6.4 Position Data Class

In [17]:


@dataclass
class Position:
    """
    Mutable position representation.

    Tracks current holdings in a symbol.

    Attributes:
        sym: Trading symbol
        signed_qty: Current quantity (positive = long, negative = short)
        price: Average entry price
    """
    sym: str
    signed_qty: Decimal
    price: Decimal

    def close(self) -> Order:
        """Generate an order to close this position."""
        return Order(self.sym, -self.signed_qty)

    def is_long(self) -> bool:
        return is_long(self.signed_qty)

    def unrealized_pnl(self, current_price: Decimal) -> Decimal:
        """Calculate unrealized P&L at current market price."""
        entry_val = self.price * self.signed_qty
        exit_val = current_price * -self.signed_qty
        return entry_val + exit_val

---
## 7. Account and Exchange APIs

### 7.1 Abstract Account Interface

In [18]:


from typing import Dict, List


class Account(ABC):
    """Abstract interface for trading accounts."""

    @abstractmethod
    def balance(self) -> Decimal:
        """Return current account balance."""
        pass

    @abstractmethod
    def get_position(self, sym: str) -> Optional[Position]:
        """Return current position for a symbol."""
        pass

### 7.2 Test Account Implementation

In [19]:


class TestAccount(Account):
    """
    Simulated account for backtesting and paper trading.

    Tracks balance, positions, and trade history.
    """

    def __init__(self, _balance: Decimal) -> None:
        self._balance = _balance
        self._positions: Dict[str, Position] = {}
        self._trades: List[Trade] = []

    def balance(self) -> Decimal:
        return self._balance

    def get_position(self, sym: str) -> Optional[Position]:
        return self._positions.get(sym)

    def __repr__(self) -> str:
        return f"TestAccount(balance={self._balance}, positions={self._positions}, trades={len(self._trades)})"


# Example
acc = TestAccount(Decimal(50.0))
print(f"Account balance: ${acc.balance()}")
print(f"Account state: {acc}")

Account balance: $50
Account state: TestAccount(balance=50, positions={}, trades=0)


### 7.3 Abstract Exchange Interface

In [20]:


class Exchange(Account):
    """
    Abstract interface for trading exchanges.

    Extends Account with order execution capabilities.
    """

    @abstractmethod
    def market_order(self, sym: str, signed_qty: Decimal, price: Decimal) -> Trade:
        """Execute a market order at the given price."""
        pass

    @abstractmethod
    def limit_order(self, sym: str, signed_qty: Decimal, price: Decimal,
                    post_only: bool = False) -> Optional[Trade]:
        """Execute a limit order (may not fill immediately)."""
        pass

### 7.4 Test Exchange Implementation

In [21]:


class TestExchange(Exchange):
    """
    Simulated exchange for testing strategies.

    Executes orders instantly at specified prices.
    """

    def __init__(self, account: TestAccount):
        self._account = account

    def market_order(self, sym: str, signed_qty: Decimal, price: Decimal) -> Trade:
        """Execute market order and update account."""
        trade = self._update_position(sym, signed_qty, price)
        self._account._balance += trade.pnl
        self._account._trades.append(trade)
        return trade

    def _update_position(self, sym: str, signed_qty: Decimal, price: Decimal) -> Trade:
        """Update position and calculate P&L."""
        position = self._account._positions.pop(sym, None)
        pnl = Decimal(0.0)

        if position is not None:
            # Closing existing position
            entry_val = position.price * position.signed_qty
            exit_val = price * position.signed_qty
            pnl = exit_val - entry_val
        else:
            # Opening new position
            self._account._positions[sym] = Position(sym, signed_qty, price)

        return Trade(sym, signed_qty, price, pnl)

    def limit_order(self, sym: str, signed_qty: Decimal, price: Decimal,
                    post_only: bool = False) -> Optional[Trade]:
        raise NotImplementedError("Limit orders not yet implemented")

    def balance(self) -> Decimal:
        return self._account.balance()

    def get_position(self, sym: str) -> Optional[Position]:
        return self._account.get_position(sym)

    def __repr__(self) -> str:
        return f"TestExchange(balance={self.balance()}, positions={self._account._positions})"

### 7.5 Exchange Example: Open and Close Position

In [22]:

# Crear exchange con cuenta
exchange = TestExchange(TestAccount(Decimal(50.0)))

# Open a long position
price = Decimal(10)
qty = Decimal(5.0)
trade = exchange.market_order('BTCUSDT', qty, price)
print(f"Opened position: {trade}")
print(f"Exchange state: {exchange}")

# Cerrar la posición a un precio más alto (¡ganancia!)
price = Decimal(15.0)
trade = exchange.market_order('BTCUSDT', -qty, price)
print(f"\nClosed position: {trade}")
print(f"Exchange state: {exchange}")

# Calculate expected P&L
entry_notional = Decimal(5) * Decimal(10)  # $50
exit_notional = Decimal(5) * Decimal(15)   # $75
expected_pnl = exit_notional - entry_notional
print(f"\nExpected P&L: ${expected_pnl} (5 × ($15 - $10))")

Opened position: Trade(LONG 5 BTCUSDT @ 10, PnL: 0)
Exchange state: TestExchange(balance=50, positions={'BTCUSDT': Position(sym='BTCUSDT', signed_qty=Decimal('5'), price=Decimal('10'))})

Closed position: Trade(SHORT 5 BTCUSDT @ 15, PnL: 25)
Exchange state: TestExchange(balance=75, positions={})

Expected P&L: $25 (5 × ($15 - $10))


---
## 8. Strategy Implementation

### 8.1 Abstract Strategy Interface

In [23]:


class Strategy(ABC):
    """Abstract interface for trading strategies."""

    @abstractmethod
    def on_tick(self, price: float, account: Account) -> Optional[List[Order]]:
        """Process new price tick and return orders (if any)."""
        pass

### 8.2 Basic Taker Strategy Implementation

In [24]:

import torch.nn as nn


class BasicTakerStrat(Strategy):
    """
    Simple taker strategy using compounding position sizing.

    On each tick:
    1. Calculate log return features
    2. Run model inference
    3. Generate orders based on prediction direction
    4. Size position based on current equity

    Args:
        sym: Trading symbol
        model: Trained PyTorch model
        log_return_lags: Feature generator
        scale_factor: Position size multiplier (default 1.0)
    """

    def __init__(self,
                 sym: str,
                 model: nn.Module,
                 log_return_lags: LogReturnLags,
                 scale_factor: Decimal = None) -> None:
        self.sym = sym
        self.model = model
        self.log_return_lags = log_return_lags
        self.scale_factor = Decimal(scale_factor) if scale_factor else Decimal(1.0)

    def _signed_compound_trade_size(
            self,
            y_hat: float,
            account: Account,
            cur_price: Decimal,
            position: Optional[Position]
    ) -> Decimal:
        """Calculate position size based on prediction and current equity."""
        dir_signal = np.sign(y_hat)
        cur_balance = account.balance()

        # Include unrealized P&L in equity calculation
        unrealized_balance = cur_balance
        if position:
            unrealized_balance += position.unrealized_pnl(cur_price)

        # Size position as percentage of equity
        qty = unrealized_balance / cur_price
        signed_qty = Decimal(dir_signal) * qty

        return signed_qty * self.scale_factor

    def _create_orders(
            self,
            y_hat: torch.Tensor,
            account: Account,
            price: Decimal
    ) -> List[Order]:
        """Generate orders based on model prediction."""
        position = account.get_position(self.sym)
        signed_trade_size = self._signed_compound_trade_size(
            y_hat.item(), account, price, position
        )

        # Create new position order
        open_order = Order(self.sym, signed_trade_size)

        # Si tenemos una posición existente, cerrarla primero
        if position is not None:
            close_order = Order(position.sym, -position.signed_qty)
            return [close_order, open_order]

        return [open_order]

    def on_tick(self, price: float, account: Account) -> List[Order]:
        """Process price tick and generate orders."""
        X = self.log_return_lags.on_tick(price)

        if X is not None:
            with torch.no_grad():
                y_hat = self.model(X)
                orders = self._create_orders(y_hat, account, Decimal(price))
                return orders

        return []

---
## 9. Strategy Execution Demo

### 9.1 Initialize Components

In [25]:

# Feature generator (3 lags)
lags = LogReturnLags(3)

# Cuenta con $100 de balance inicial
acc = TestAccount(Decimal(100.0))

# Estrategia usando nuestro modelo entrenado
strat = BasicTakerStrat('BTCUSDT', model, lags, Decimal(1.0))

### 9.2 Alimentar Precios Iniciales (Período de Calentamiento)

Necesitamos 4 precios para generar 3 log returns (features del modelo).

In [26]:

print("=== Warm-up Period ===")

# First 12h interval
orders = strat.on_tick(10.0, acc)
print(f"Price 10.0: orders = {orders}")

# Second 12h interval
orders = strat.on_tick(120.0, acc)
print(f"Price 120.0: orders = {orders}")

# Third 12h interval
orders = strat.on_tick(90.0, acc)
print(f"Price 90.0: orders = {orders}")

# Fourth 12h interval (first trade!)
orders = strat.on_tick(100, acc)
print(f"Price 100.0: orders = {orders}")

=== Warm-up Period ===
Price 10.0: orders = []
Price 120.0: orders = []
Price 90.0: orders = []
Price 100.0: orders = [Order(sym='BTCUSDT', signed_qty=Decimal('-1'))]


### 9.3 Execute First Trade

In [27]:

print("\n=== Execute First Trade ===")

exchange = TestExchange(acc)
order = orders[0]
trade = exchange.market_order(order.sym, order.signed_qty, Decimal(100))
print(f"Executed: {trade}")
print(f"Exchange: {exchange}")


=== Execute First Trade ===
Executed: Trade(SHORT 1 BTCUSDT @ 100, PnL: 0)
Exchange: TestExchange(balance=100, positions={'BTCUSDT': Position(sym='BTCUSDT', signed_qty=Decimal('-1'), price=Decimal('100'))})


### 9.4 Continue Trading

In [28]:

print("\n=== Next Tick ===")

orders = strat.on_tick(115, acc)
print(f"Price 115.0: orders = {orders}")

# Close previous position
order = orders[0]
trade = exchange.market_order(order.sym, order.signed_qty, Decimal(115))
print(f"Closed: {trade}")
print(f"Exchange: {exchange}")

# Open new position
order = orders[1]
trade = exchange.market_order(order.sym, order.signed_qty, Decimal(115))
print(f"Opened: {trade}")
print(f"Exchange: {exchange}")


=== Next Tick ===
Price 115.0: orders = [Order(sym='BTCUSDT', signed_qty=Decimal('1')), Order(sym='BTCUSDT', signed_qty=Decimal('-1'))]
Closed: Trade(LONG 1 BTCUSDT @ 115, PnL: -15)
Exchange: TestExchange(balance=85, positions={})
Opened: Trade(SHORT 1 BTCUSDT @ 115, PnL: 0)
Exchange: TestExchange(balance=85, positions={'BTCUSDT': Position(sym='BTCUSDT', signed_qty=Decimal('-1'), price=Decimal('115'))})


### 9.5 Another Trade

In [29]:

print("\n=== Another Tick ===")

orders = strat.on_tick(100, acc)
print(f"Price 100.0: orders = {orders}")

# Close and open
for order in orders:
    trade = exchange.market_order(order.sym, order.signed_qty, Decimal(100))
    print(f"Executed: {trade}")

print(f"\nFinal Exchange State: {exchange}")


=== Another Tick ===
Price 100.0: orders = [Order(sym='BTCUSDT', signed_qty=Decimal('1')), Order(sym='BTCUSDT', signed_qty=Decimal('0.7'))]
Executed: Trade(LONG 1 BTCUSDT @ 100, PnL: 15)
Executed: Trade(LONG 0.7 BTCUSDT @ 100, PnL: 0)

Final Exchange State: TestExchange(balance=100, positions={'BTCUSDT': Position(sym='BTCUSDT', signed_qty=Decimal('0.7'), price=Decimal('100'))})


---
## Summary

En este notebook, construimos un sistema de trading en tiempo real completo:

### 1. Streaming Data Structures
- **Tick interface**: Composable streaming processors
- **Sliding windows**: DequeWindow and NumpyWindow
- **LogReturnLags**: Streaming feature generator

### 2. Trading System Components
- **Order/Trade/Position**: Immutable data classes
- **Account**: Balance and position tracking
- **Exchange**: Order execution simulation

### 3. Strategy Implementation
- **BasicTakerStrat**: Compounding position sizing
- Real-time model inference
- Order generation based on predictions

### Production Considerations

Para un sistema en producción, también necesitarías:
- **Conexiones WebSocket** para feeds de precios en tiempo real
- **Integración de REST API** para ejecución de órdenes
- **Error handling** and retry logic
- **Logging** and monitoring
- **Risk limits** and circuit breakers
- **Optimización de latencia** (considerar Rust para hot paths)

---

## Series Complete!

A lo largo de tres notebooks, construimos:

1. **Parte 1**: Modelo ML para predecir log returns
2. **Parte 2**: Estrategia de trading con dimensionamiento y apalancamiento
3. **Parte 3**: Implementación en tiempo real architecture

**Next steps:**
- Connect to a real exchange API (Binance, Bybit, etc.)
- Implement paper trading mode
- Add proper risk management
- Monitor performance and alpha decay